#### Imports

In [ ]:
# Install required libraries (uncomment and run if needed)
!pip install stim pymatching numpy matplotlib torch tqdm networkx
!pip install torch_geometric

# For CUDA support with PyTorch Geometric, you may need:
# !pip install torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

In [ ]:
import sys
from pathlib import Path

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('..')

sys.path.insert(0, str(BASE_PATH))

# Common imports
import stim
import pymatching
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree
from datetime import datetime
from tqdm import tqdm
import networkx as nx
import os

# Import model classes and utilities from models.py
from models import (
    surface_code_circuit,
    count_logical_errors,
    ler_mwpm,
    plot_mwpm,
    SurfaceCodeSampler,
    SparseGraph,
    visualize_sparse_graph,
    GATModel,
    GAT,
)

In [ ]:
# If CUDA is available, show more details
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"Current device: {torch.cuda.current_device()}")
else:
    print("PyTorch is using CPU only")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

#### Training

##### Definition

In [ ]:
class GATTrainer:
    """
    Highly customizable trainer for GAT models on quantum error correction data.
    
    This trainer handles the entire training pipeline:
    - Generates samples across multiple code distances and error rates
    - Converts samples to sparse graph representations
    - Trains the GAT model
    - Returns the trained model
    
    Supports mixed-distance training where samples can be drawn from different
    code distances with configurable proportions.
    
    Attributes:
        nickname (str): Human-readable name for the model
        distances (list[int]): Code distances to train on
        distance_weights (list[float]): Proportion of samples per distance
        p_values (list[float]): Physical error rates
        p_weights (list[float]): Distribution of error rates
        sample_size (int): Total number of training samples
        epochs (int): Number of training epochs
        batch_size (int): Training batch size
        lr (float): Learning rate
        hidden_dim (int): GAT hidden dimension
        num_layers (int): Number of GAT layers
        heads (int): Number of attention heads
        dropout (float): Dropout rate for attention weights
        k_neighbors (int): K-neighbors for SparseGraph
        device (torch.device): Compute device
    """
    
    def __init__(
        self,
        nickname: str,
        distances: list[int],
        distance_weights: list[float],
        p_values: list[float],
        p_weights: list[float],
        sample_size: int,
        epochs: int = 10,
        batch_size: int = 64,
        lr: float = 1e-3,
        hidden_dim: int = 128,
        num_layers: int = 4,
        heads: int = 4,
        dropout: float = 0.0,
        k_neighbors: int = 6,
        device: torch.device = None,
        base_path: Path = None
    ):
        """
        Initialize the GAT trainer.
        
        Args:
            nickname: Human-readable name for the model
            distances: Code distances to train on, e.g., [3, 5, 7]
            distance_weights: Proportion for each distance, must sum to 1.0
            p_values: Physical error rates to sample from
            p_weights: Distribution of error rates, must sum to 1.0
            sample_size: Total number of training samples to generate
            epochs: Number of training epochs (default: 10)
            batch_size: Training batch size (default: 64)
            lr: Learning rate (default: 1e-3)
            hidden_dim: GAT hidden dimension (default: 128)
            num_layers: Number of GAT layers (default: 4)
            heads: Number of attention heads (default: 4)
            dropout: Dropout rate for attention weights (default: 0.0)
            k_neighbors: K-neighbors for SparseGraph (default: 6)
            device: Compute device (defaults to CUDA if available)
            base_path: Base path for model storage (defaults to current directory)
        
        Raises:
            ValueError: If distances and distance_weights have different lengths
            ValueError: If p_values and p_weights have different lengths
            ValueError: If distance_weights don't sum to 1.0
            ValueError: If p_weights don't sum to 1.0
        """
        # Validate inputs
        if len(distances) != len(distance_weights):
            raise ValueError(f"distances and distance_weights must have same length. "
                           f"Got {len(distances)} and {len(distance_weights)}")
        
        if len(p_values) != len(p_weights):
            raise ValueError(f"p_values and p_weights must have same length. "
                           f"Got {len(p_values)} and {len(p_weights)}")
        
        weight_sum = sum(distance_weights)
        if not np.isclose(weight_sum, 1.0, atol=1e-6):
            raise ValueError(f"distance_weights must sum to 1.0, got {weight_sum}")
        
        p_weight_sum = sum(p_weights)
        if not np.isclose(p_weight_sum, 1.0, atol=1e-6):
            raise ValueError(f"p_weights must sum to 1.0, got {p_weight_sum}")
        
        # Store configuration
        self.nickname = nickname
        self.distances = distances
        self.distance_weights = distance_weights
        self.p_values = p_values
        self.p_weights = p_weights
        self.sample_size = sample_size
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.heads = heads
        self.dropout = dropout
        self.k_neighbors = k_neighbors
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.base_path = base_path
        
        print(f"GATTrainer initialized:")
        print(f"  Nickname: {nickname}")
        print(f"  Distances: {distances} (weights: {distance_weights})")
        print(f"  Error rates: {p_values} (weights: {p_weights})")
        print(f"  Sample size: {sample_size:,}")
        print(f"  Epochs: {epochs}, Batch size: {batch_size}, LR: {lr}")
        print(f"  Model: hidden_dim={hidden_dim}, num_layers={num_layers}, heads={heads}, dropout={dropout}")
        print(f"  Device: {self.device}")
    
    def train(self, verbose: bool = True) -> GAT:
        """
        Execute the full training pipeline.
        
        This method:
        1. Initializes the sampler and graph builder
        2. Generates samples for each distance according to weights
        3. Converts all samples to sparse graphs
        4. Trains the GAT model
        5. Returns the trained model
        
        Args:
            verbose: Print progress information (default: True)
        
        Returns:
            GAT: The trained GAT model instance
        """
        import random
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"Starting training pipeline for: {self.nickname}")
            print(f"{'='*60}")
        
        # Initialize sampler and graph builder
        sampler = SurfaceCodeSampler(p=self.p_values[0], device=self.device)
        graph_builder = SparseGraph(k_neighbors=self.k_neighbors, device=self.device)
        
        # Generate samples for each distance
        all_graphs = []
        
        for i, (d, weight) in enumerate(zip(self.distances, self.distance_weights)):
            # Calculate number of samples for this distance
            if i == len(self.distances) - 1:
                # Last distance gets remaining samples to handle rounding
                n_samples = self.sample_size - len(all_graphs)
            else:
                n_samples = int(round(self.sample_size * weight))
            
            if n_samples <= 0:
                continue
            
            if verbose:
                print(f"\nGenerating {n_samples:,} samples for distance d={d}...")
            
            # Generate samples with the specified error rate distribution
            detections, labels = sampler.sample(
                d=d,
                num_samples=n_samples,
                p_values=self.p_values,
                p_weights=self.p_weights
            )
            
            if verbose:
                print(f"  Converting to sparse graphs...")
            
            # Convert to graphs with progress bar
            graphs = []
            with tqdm(total=n_samples, desc=f"  Converting d={d}", 
                     unit="graph", leave=True, dynamic_ncols=True) as pbar:
                for j in range(detections.shape[0]):
                    graph = graph_builder.to_pyg(detections[j], labels[j])
                    graphs.append(graph)
                    pbar.update(1)
            
            all_graphs.extend(graphs)
            
            if verbose:
                print(f"  Generated {len(graphs):,} graphs (total: {len(all_graphs):,})")
        
        # Shuffle the combined dataset
        if verbose:
            print(f"\nShuffling {len(all_graphs):,} graphs...")
        random.shuffle(all_graphs)
        
        # Initialize the GAT model
        if verbose:
            print(f"\nInitializing GAT model...")
        
        gat = GAT(
            nickname=self.nickname,
            in_channels=5,  # SparseGraph outputs 5 features
            hidden_dim=self.hidden_dim,
            num_layers=self.num_layers,
            heads=self.heads,
            dropout=self.dropout,
            device=self.device,
            base_path=self.base_path
        )
        
        # Train the model
        if verbose:
            print(f"\nStarting training...")
        
        gat.train(
            graphs=all_graphs,
            epochs=self.epochs,
            batch_size=self.batch_size,
            lr=self.lr,
            verbose=verbose
        )
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"Training complete for: {self.nickname}")
            print(f"{'='*60}")
        
        return gat
    
    def __repr__(self) -> str:
        return (f"GATTrainer(nickname='{self.nickname}', "
                f"distances={self.distances}, "
                f"sample_size={self.sample_size:,}, "
                f"epochs={self.epochs})")

##### Visualizing

In [ ]:
# # Initialize sampler with default p=0.005
# sampler = SurfaceCodeSampler(p=0.005, device=device)

# # Define the distance to use for this training run
# d = 3

# # Define the error rates we want to sample from
# p_values = [0.001, 0.002, 0.003, 0.004, 0.005]

# # Evenly distributed weights (20% each)
# p_weights = [0.2, 0.2, 0.2, 0.2, 0.2]

# # Generate training data with 10,000 samples
# num_samples = 10000 #00
# train_detections, train_labels, p_indices = sampler.sample(
#     d=d,
#     num_samples=num_samples,
#     p_values=p_values,
#     p_weights=p_weights,
#     return_p_labels=True
# )

# # Print dataset info
# print(f"\nTraining data generated:")
# print(f"  Detections shape: {train_detections.shape}")
# print(f"  Labels shape: {train_labels.shape}")
# print(f"  Labels distribution: {train_labels.sum().item():.0f} positive / {num_samples} total")

# # Verify distribution of samples across error rates
# print(f"\nSamples per error rate:")
# for i, p in enumerate(p_values):
#     count = (p_indices == i).sum().item()
#     print(f"  p={p}: {count} samples ({count/num_samples*100:.1f}%)")

# # Generate graphs using SparseGraph
# graph_builder = SparseGraph(k_neighbors=6, device=device)
# graphs = graph_builder.batch_to_pyg(train_detections, train_labels)

# # Find the first non-empty graph for visualization
# for i, g in enumerate(graphs):
#     if g.x.shape[0] > 0:
#         print(f"Visualizing graph from sample {i}")
#         visualize_sparse_graph(g, title=f"Sample {i} - Sparse Graph")
#         break

##### Logic

In [ ]:
# Directory where models are saved
models_dir = BASE_PATH / "models" / "gat"

# Distances to train
distances = [3, 5, 7, 9]

# Store trained models
trained_models = {}

for d in distances:
    nickname = f"d{d}_baseline"
    
    # Check if model already exists (any file starting with nickname_)
    existing = list(models_dir.glob(f"{nickname}_*.pt"))
    if existing:
        print(f"Model '{nickname}' already exists: {existing[0].name}, skipping...")
        continue
    
    print(f"\n{'='*60}")
    print(f"Training {nickname}...")
    print(f"{'='*60}")
    
    trainer = GATTrainer(
        nickname=nickname,
        distances=[d],
        distance_weights=[1.0],
        p_values=[0.001, 0.003, 0.005, 0.007],
        p_weights=[0.25, 0.25, 0.25, 0.25],
        sample_size=10**6,
        epochs=10,
        batch_size=128,
        lr=1e-3,
        hidden_dim=128,
        num_layers=4,
        heads=4,
        dropout=0.0,
        base_path=BASE_PATH
    )
    
    trained_models[nickname] = trainer.train()

# Save all trained models
print(f"\n{'='*60}")
print(f"Saving {len(trained_models)} trained models...")
print(f"{'='*60}")

for nickname, model in trained_models.items():
    model.save(nickname)

print(f"\nDone! Trained and saved {len(trained_models)} models.")